In [1]:
from pathlib import Path
import types
import dill


# ============================================================
# LOCALISATION AUTOMATIQUE DU PROJET
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

candidats = [
    CURRENT_DIR,
    CURRENT_DIR / "Script_intermediaire",
    Path.home() / "Downloads" / "Projet 4" / "Script_intermediaire",
]

# Ajout des dossiers parents et de leurs éventuels sous-dossiers
for parent in CURRENT_DIR.parents:
    candidats.append(parent)

    candidats.append(parent / "Script_intermediaire")


SCRIPT_DIR = next(
    (
        chemin
        for chemin in candidats
        if chemin.is_dir()
        and chemin.name == "Script_intermediaire"
    ),
    None,
)


if SCRIPT_DIR is None:
    raise FileNotFoundError(
        "Impossible de localiser le dossier 'Script_intermediaire'.\n"
        f"Dossier courant détecté : {CURRENT_DIR}"
    )


SESSION_DIR = SCRIPT_DIR / "outputs"
SESSION_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# OBJETS TECHNIQUES À NE PAS SAUVEGARDER
# ============================================================

NOMS_TECHNIQUES = {
    "In",
    "Out",
    "exit",
    "quit",
    "get_ipython",
    "open",
    "CURRENT_DIR",
    "SCRIPT_DIR",
    "SESSION_DIR",
    "NOMS_TECHNIQUES",
    "candidats",
    "parent",
    "chemin",
    "sauvegarder_session",
    "charger_session",
    "objets_restaures",
}


# ============================================================
# SAUVEGARDE AUTOMATIQUE
# ============================================================

def sauvegarder_session(etape):
    """
    Sauvegarde automatiquement tous les objets utilisateur
    sérialisables présents dans le notebook.

    Les modules, objets internes de Jupyter/VS Code et objets
    non sérialisables sont ignorés.
    """

    chemin_final = SESSION_DIR / f"session_etape{etape}.pkl"
    chemin_temporaire = SESSION_DIR / f"session_etape{etape}.tmp"

    objets_serialises = {}
    objets_ignores = {}

    for nom, objet in list(globals().items()):

        # Objets internes ou privés
        if nom.startswith("_"):
            continue

        # Infrastructure de sauvegarde
        if nom in NOMS_TECHNIQUES:
            continue

        # Objets techniques injectés par Data Wrangler
        if "DW_OUTPUT_FORMATTER" in nom:
            continue

        # Les modules seront réimportés normalement
        if isinstance(objet, types.ModuleType):
            continue

        try:
            # Chaque objet est sérialisé séparément.
            # Un objet défectueux ne bloque donc pas tous les autres.
            objets_serialises[nom] = dill.dumps(
                objet,
                recurse=True
            )

        except Exception as erreur:
            objets_ignores[nom] = (
                f"{type(erreur).__name__}: {erreur}"
            )

    contenu_session = {
        "version": 1,
        "etape": etape,
        "objets": objets_serialises,
    }

    # Écriture atomique : on écrit d'abord dans un fichier temporaire
    with open(chemin_temporaire, "wb") as fichier:
        dill.dump(contenu_session, fichier)

    chemin_temporaire.replace(chemin_final)

    print("=" * 65)
    print(f"Session de l'étape {etape} enregistrée")
    print(f"Fichier : {chemin_final}")
    print(f"Objets sauvegardés : {len(objets_serialises)}")
    print(f"Objets ignorés : {len(objets_ignores)}")

    if objets_ignores:
        print("\nObjets non sérialisables ignorés :")

        for nom, erreur in objets_ignores.items():
            print(f" - {nom} → {erreur}")

    print("=" * 65)

    return chemin_final


# ============================================================
# CHARGEMENT AUTOMATIQUE
# ============================================================

def charger_session(etape):
    """
    Recharge automatiquement tous les objets enregistrés
    à l'étape demandée, avec leurs noms d'origine.
    """

    chemin_session = SESSION_DIR / f"session_etape{etape}.pkl"

    if not chemin_session.exists():
        raise FileNotFoundError(
            f"Session introuvable : {chemin_session}\n"
            f"Exécute d'abord entièrement le notebook de l'étape {etape}."
        )

    with open(chemin_session, "rb") as fichier:
        contenu_session = dill.load(fichier)

    objets_serialises = contenu_session["objets"]

    objets_restaures = {}
    objets_non_restaures = {}

    for nom, objet_serialise in objets_serialises.items():

        try:
            objets_restaures[nom] = dill.loads(
                objet_serialise
            )

        except Exception as erreur:
            objets_non_restaures[nom] = (
                f"{type(erreur).__name__}: {erreur}"
            )

    globals().update(objets_restaures)

    print("=" * 65)
    print(f"Session de l'étape {etape} chargée")
    print(f"Fichier : {chemin_session}")
    print(f"Objets restaurés : {len(objets_restaures)}")
    print(f"Objets non restaurés : {len(objets_non_restaures)}")

    if objets_non_restaures:
        print("\nObjets impossibles à restaurer :")

        for nom, erreur in objets_non_restaures.items():
            print(f" - {nom} → {erreur}")

    print("=" * 65)

    return objets_restaures


print("Infrastructure de session initialisée")
print("Dossier des notebooks :", SCRIPT_DIR)
print("Dossier des sessions :", SESSION_DIR)

Infrastructure de session initialisée
Dossier des notebooks : /Users/thibaultdautheville/Downloads/Projet 4/Script_intermediaire
Dossier des sessions : /Users/thibaultdautheville/Downloads/Projet 4/Script_intermediaire/outputs


In [2]:
objets_restaures = charger_session(2)

Session de l'étape 2 chargée
Fichier : /Users/thibaultdautheville/Downloads/Projet 4/Script_intermediaire/outputs/session_etape2.pkl
Objets restaurés : 85
Objets non restaurés : 0


# Etape 3 Réalisation d'un premier modèle de classification

modéle étaloon pour mesurer les efforts des modèles et optimisation ultérieures

Avoir un point de comparaison, un modèle plus compelxe n'est forcément plus performant


La variable cible a été retravaillée et transformée en binaire selon la logique : fusion des modalité 1+2 → "insatisfait" (0) et 3+4 → "satisfait" (1)

In [3]:
# Binarisation explicite avec règle nommée
y_binaire = y.map({1: 0, 2: 0, 3: 1, 4: 1})

# Vérification immédiate contre la source
print("Distribution y_binaire :")
print(y_binaire.value_counts())
print(y_binaire.value_counts(normalize=True))

# Split stratifié
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binaire, test_size=0.2, random_state=42, stratify=y_binaire
)

print("\nTrain :", y_train.value_counts(normalize=True).to_dict())
print("Test  :", y_test.value_counts(normalize=True).to_dict())

Distribution y_binaire :
satisfaction_employee_environnement
1    899
0    571
Name: count, dtype: int64
satisfaction_employee_environnement
1    0.611565
0    0.388435
Name: proportion, dtype: float64

Train : {1: 0.6113945578231292, 0: 0.38860544217687076}
Test  : {1: 0.6122448979591837, 0: 0.3877551020408163}


### méthode Dummy

Cette méthode va reprendre les propotions des classes dans le jeu de train et va décider aléatoireement de la classe accoée à chaque observation du jeu de test en utilisant ces mêmes proportions.

In [4]:


dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)


y_pred_train_dummy = dummy.predict(X_train)
y_pred_test_dummy = dummy.predict(X_test)

print("=== TRAIN ===")
print("Accuracy :", accuracy_score(y_train, y_pred_train_dummy))
print(classification_report(y_train, y_pred_train_dummy))

print("=== TEST ===")
print("Accuracy :", accuracy_score(y_test, y_pred_test_dummy))
print(classification_report(y_test, y_pred_test_dummy))

print("Matrice de confusion (test) :\n", confusion_matrix(y_test, y_pred_test_dummy))

=== TRAIN ===
Accuracy : 0.6113945578231292
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       457
           1       0.61      1.00      0.76       719

    accuracy                           0.61      1176
   macro avg       0.31      0.50      0.38      1176
weighted avg       0.37      0.61      0.46      1176

=== TEST ===
Accuracy : 0.6122448979591837
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       114
           1       0.61      1.00      0.76       180

    accuracy                           0.61       294
   macro avg       0.31      0.50      0.38       294
weighted avg       0.37      0.61      0.46       294

Matrice de confusion (test) :
 [[  0 114]
 [  0 180]]


/opt/anaconda3/envs/projet4/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/projet4/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/projet4/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

0,61 pour l'accuracy test représente bien la proportion de la classe majoritaire dans le test (61,22%)

In [5]:
y_train.value_counts()

satisfaction_employee_environnement
1    719
0    457
Name: count, dtype: int64

### RegressionLogistique

In [6]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)


y_pred_train_logreg = logreg.predict(X_train_scaled)
y_pred_test_logreg = logreg.predict(X_test_scaled)

print("=== TRAIN ===")
print("Accuracy :", accuracy_score(y_train, y_pred_train_logreg))
print(classification_report(y_train, y_pred_train_logreg))

print("=== TEST ===")
print("Accuracy :", accuracy_score(y_test, y_pred_test_logreg))
print(classification_report(y_test, y_pred_test_logreg))

print("Matrice de confusion (test) :\n", confusion_matrix(y_test, y_pred_test_logreg))

=== TRAIN ===
Accuracy : 0.6326530612244898
              precision    recall  f1-score   support

           0       0.58      0.20      0.30       457
           1       0.64      0.91      0.75       719

    accuracy                           0.63      1176
   macro avg       0.61      0.55      0.52      1176
weighted avg       0.62      0.63      0.57      1176

=== TEST ===
Accuracy : 0.6122448979591837
              precision    recall  f1-score   support

           0       0.50      0.13      0.21       114
           1       0.62      0.92      0.74       180

    accuracy                           0.61       294
   macro avg       0.56      0.52      0.48       294
weighted avg       0.58      0.61      0.54       294

Matrice de confusion (test) :
 [[ 15  99]
 [ 15 165]]


### XGBoost

In [7]:
print(y_train.unique())
print(y_test.unique())

[1 0]
[1 0]


In [8]:
y_binaire.value_counts()

satisfaction_employee_environnement
1    899
0    571
Name: count, dtype: int64

In [9]:


xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_scaled, y_train)

y_pred_train_xgb = xgb_model.predict(X_train_scaled)
y_pred_test_xgb = xgb_model.predict(X_test_scaled)

print("=== TRAIN ===")
print("Accuracy :", accuracy_score(y_train, y_pred_train_xgb))
print(classification_report(y_train, y_pred_train_xgb))

print("=== TEST ===")
print("Accuracy :", accuracy_score(y_test, y_pred_test_xgb))
print(classification_report(y_test, y_pred_test_xgb))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred_test_xgb))

=== TRAIN ===
Accuracy : 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       457
           1       1.00      1.00      1.00       719

    accuracy                           1.00      1176
   macro avg       1.00      1.00      1.00      1176
weighted avg       1.00      1.00      1.00      1176

=== TEST ===
Accuracy : 0.608843537414966
              precision    recall  f1-score   support

           0       0.49      0.38      0.43       114
           1       0.66      0.76      0.70       180

    accuracy                           0.61       294
   macro avg       0.58      0.57      0.57       294
weighted avg       0.59      0.61      0.60       294

Matrice de confusion :
 [[ 43  71]
 [ 44 136]]


Récapitulatif des trois modèles


y = satisfaction_employee_environnement en binaire, satisfait 1, insatisfait 0



| Métrique | Dummy | LogReg scaled | XGBoost | Verdict |
|---|---|---|---|---|
| Accuracy train | 0.6114 | 0.6327 | **1.0000** | XGBoost mémorise, ne généralise pas |
| Accuracy test | 0.6122 | 0.6122 | 0.5748 | **XGBoost est le pire des trois en généralisation** |
| Écart train-test (accuracy) | 0.0008 | 0.0204 | **0.4252** | Overfitting massif confirmé pour XGBoost, quasi nul ailleurs |
| Precision classe 0 (insatisfaits) | 0.00 | 0.50 | 0.44 | LogReg légèrement meilleure quand elle prédit "insatisfait" |
| Recall classe 0 (insatisfaits) | 0.00 | **0.13** | 0.35 | **XGBoost détecte mieux les insatisfaits que LogReg** — inverse de ce que j'avais dit avant |
| F1 classe 0 | 0.00 | 0.21 | **0.39** | XGBoost gagne ici malgré son overfitting global |
| Recall classe 1 (satisfaits) | 1.00 | 0.92 | 0.72 | Dummy et LogReg sur-privilégient la classe majoritaire |
| F1 macro test | 0.38 | 0.48 | **0.53** | XGBoost meilleur en moyenne macro, malgré l'overfitting train |
| F1 weighted test | 0.46 | 0.54 | **0.56** | Confirme XGBoost légèrement devant en usage global |




- Aucun des trois modèles n'est utilisable pour une action RH
- XGBoost overfit énormément (std de 0,42)
- aucun modèle n'est bon

### Deuxième modélisation


Après avoir fait un premier features enegeneering et premières modélisation, je me demande si la valeur cible ici 'satisfaction_employee_environnement' ne serait pas mauvaise, et si il faudrait privilégier une autre var type 'a_quitte_l_entreprise'

16% des employés ont quitté l'entreprise. Mais cette variable est elle enregistrée avant, pendant ou après les autres ? Il faut vérifier l'absence de fuite de donnée.

Le dataset ne fournit pas de timesplan, mais l'on peut mesurer l'ancientté moyenne, est elle significativement plus élevé des groupe des Oui ou des Non qui ont quitté l'entreprise?

In [10]:
print(DataFrameRH['a_quitte_l_entreprise'].value_counts())
print()
print(DataFrameRH['a_quitte_l_entreprise'].value_counts(normalize=True).round(4) * 100)

a_quitte_l_entreprise
0    1233
1     237
Name: count, dtype: int64

a_quitte_l_entreprise
0    83.88
1    16.12
Name: proportion, dtype: float64


In [11]:
variables_a_risque = [
    'annees_dans_l_entreprise',
    'note_evaluation_actuelle',
    'annees_depuis_la_derniere_promotion',
    'revenu_mensuel',
    'nombre_employee_sous_responsabilite'
]

for var in variables_a_risque:
    print(f"--- {var} ---")
    print(DataFrameRH.groupby('a_quitte_l_entreprise')[var].describe()[['mean', '50%', 'std']])
    print()

--- annees_dans_l_entreprise ---
                           mean  50%       std
a_quitte_l_entreprise                         
0                      7.369019  6.0  6.096298
1                      5.130802  3.0  5.949984

--- note_evaluation_actuelle ---
                           mean  50%       std
a_quitte_l_entreprise                         
0                      3.153285  3.0  0.360408
1                      3.156118  3.0  0.363735

--- annees_depuis_la_derniere_promotion ---
                           mean  50%       std
a_quitte_l_entreprise                         
0                      2.234388  1.0  3.234762
1                      1.945148  1.0  3.153077

--- revenu_mensuel ---
                              mean     50%          std
a_quitte_l_entreprise                                  
0                      6832.739659  5204.0  4818.208001
1                      4787.092827  3202.0  3640.210367

--- nombre_employee_sous_responsabilite ---
                       mean  50

### Modélisation avec a_quitte_l_entreprise

nouvelle cible : a_quitte_l_entreprise en binaire



In [12]:
#Split second

y2 = DataFrameRH['a_quitte_l_entreprise']
X2 = DataFrameRH.drop(columns='a_quitte_l_entreprise')

# Split stratifié — stratify=y2 garantit que le déséquilibre 84/16 est respecté
# à l'identique dans train et test (sinon tu pourrais avoir un test set encore plus déséquilibré par hasard)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

print("Train :", y2_train.value_counts(normalize=True).to_dict())
print("Test  :", y2_test.value_counts(normalize=True).to_dict())

Train : {0: 0.8384353741496599, 1: 0.16156462585034015}
Test  : {0: 0.8401360544217688, 1: 0.1598639455782313}


### Dummy

In [13]:
dummy2 = DummyClassifier(strategy='most_frequent', random_state=42)
dummy2.fit(X2_train, y2_train)

y2_pred_train_dummy = dummy2.predict(X2_train)
y2_pred_test_dummy = dummy2.predict(X2_test)

print("=== TRAIN ===")
print("Accuracy :", accuracy_score(y2_train, y2_pred_train_dummy))
print(classification_report(y2_train, y2_pred_train_dummy))

print("=== TEST ===")
print("Accuracy :", accuracy_score(y2_test, y2_pred_test_dummy))
print(classification_report(y2_test, y2_pred_test_dummy))

print("Matrice de confusion (test) :\n", confusion_matrix(y2_test, y2_pred_test_dummy))

=== TRAIN ===
Accuracy : 0.8384353741496599
              precision    recall  f1-score   support

           0       0.84      1.00      0.91       986
           1       0.00      0.00      0.00       190

    accuracy                           0.84      1176
   macro avg       0.42      0.50      0.46      1176
weighted avg       0.70      0.84      0.76      1176

=== TEST ===
Accuracy : 0.8401360544217688
              precision    recall  f1-score   support

           0       0.84      1.00      0.91       247
           1       0.00      0.00      0.00        47

    accuracy                           0.84       294
   macro avg       0.42      0.50      0.46       294
weighted avg       0.71      0.84      0.77       294

Matrice de confusion (test) :
 [[247   0]
 [ 47   0]]


/opt/anaconda3/envs/projet4/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/projet4/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/projet4/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

### Regression logistique

In [14]:
scaler2 = StandardScaler()
X2_train_scaled = scaler2.fit_transform(X2_train)
X2_test_scaled = scaler2.transform(X2_test)

logreg2 = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
logreg2.fit(X2_train_scaled, y2_train)

y2_pred_train_logreg = logreg2.predict(X2_train_scaled)
y2_pred_test_logreg = logreg2.predict(X2_test_scaled)

print("=== TRAIN ===")
print("Accuracy :", accuracy_score(y2_train, y2_pred_train_logreg))
print(classification_report(y2_train, y2_pred_train_logreg))

print("=== TEST ===")
print("Accuracy :", accuracy_score(y2_test, y2_pred_test_logreg))
print(classification_report(y2_test, y2_pred_test_logreg))

print("Matrice de confusion (test) :\n", confusion_matrix(y2_test, y2_pred_test_logreg))

=== TRAIN ===
Accuracy : 0.782312925170068
              precision    recall  f1-score   support

           0       0.96      0.78      0.86       986
           1       0.41      0.82      0.55       190

    accuracy                           0.78      1176
   macro avg       0.68      0.80      0.70      1176
weighted avg       0.87      0.78      0.81      1176

=== TEST ===
Accuracy : 0.782312925170068
              precision    recall  f1-score   support

           0       0.93      0.81      0.86       247
           1       0.39      0.66      0.49        47

    accuracy                           0.78       294
   macro avg       0.66      0.73      0.68       294
weighted avg       0.84      0.78      0.80       294

Matrice de confusion (test) :
 [[199  48]
 [ 16  31]]


### XGBoost


Il y a cependant un probleme de déséquilibre de classe (16% contre 84%). C'est un probleme de signal caché par bruit majoritaire. pour régler ce déséquilibre il faut utiliser class_weight='balanced' (déjà mis dans le code LogReg précédemment) : cette commande dit à l'algorithme "une erreur sur la classe minoritaire (16%) coûte plus cher qu'une erreur sur la majoritaire". Ça force le modèle à faire attention aux départs, pas juste à ignorer.

Même logique avec XGboost : utilisation de scale_pos_weight



In [15]:
from xgboost import XGBClassifier

# scale_pos_weight = ratio classe négative / classe positive
# Calcul automatique : compense le déséquilibre 84/16 directement dans l'algorithme
ratio2 = (y2_train == 0).sum() / (y2_train == 1).sum()

xgb_model2 = XGBClassifier(
    random_state=42, 
    eval_metric='logloss',
    scale_pos_weight=ratio2
)
xgb_model2.fit(X2_train_scaled, y2_train)

y2_pred_train_xgb = xgb_model2.predict(X2_train_scaled)
y2_pred_test_xgb = xgb_model2.predict(X2_test_scaled)

print("=== TRAIN ===")
print("Accuracy :", accuracy_score(y2_train, y2_pred_train_xgb))
print(classification_report(y2_train, y2_pred_train_xgb))

print("=== TEST ===")
print("Accuracy :", accuracy_score(y2_test, y2_pred_test_xgb))
print(classification_report(y2_test, y2_pred_test_xgb))

print("Matrice de confusion (test) :\n", confusion_matrix(y2_test, y2_pred_test_xgb))

=== TRAIN ===
Accuracy : 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       986
           1       1.00      1.00      1.00       190

    accuracy                           1.00      1176
   macro avg       1.00      1.00      1.00      1176
weighted avg       1.00      1.00      1.00      1176

=== TEST ===
Accuracy : 0.8503401360544217
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       247
           1       0.55      0.34      0.42        47

    accuracy                           0.85       294
   macro avg       0.72      0.64      0.67       294
weighted avg       0.83      0.85      0.84       294

Matrice de confusion (test) :
 [[234  13]
 [ 31  16]]


Tableau récapitulatif des résultats

Y = a_quitte_l_entreprise

| Métrique | Dummy | LogReg (class_weight=balanced) | XGBoost (scale_pos_weight) |
|---|---|---|---|
| Accuracy train | 0.8384 | 0.7823 | **1.0000** |
| Accuracy test | 0.8401 | 0.7823 | 0.8503 |
| Écart train-test | 0.0017 | **0.0000** | **0.1497** |
| Precision classe 1 (départs) | 0.00 | 0.39 | **0.56** |
| Recall classe 1 (départs) | 0.00 | **0.66** | 0.32 |
| F1 classe 1 | 0.00 | **0.49** | 0.41 |
| Recall classe 0 (restent) | 1.00 | 0.81 | **0.95** |
| F1 macro test | 0.46 | **0.68** | 0.66 |
| F1 weighted test | 0.77 | 0.80 | **0.83** |

Commentaires :


- Le dummy confirme son poste de plancher comparatif, avec un Recall de 0, il prédit personne ne part pour tout le monde, si un modèle fait moins bien que ça sur le Recall class 1, alors il est inutile.

- Accuracy test XBoost (O,85) > à celui du LogReg (0,78), le XGBoost semble meilleur. Mais le Recall Class 1 du XGBoost est inférieur à celui du LogReg d'un facteur presque 2 fois moins. Le LogReg malgré une accuracy inférieure détecte 2 départs / 3.

- XGBoost sur apprend avec un score de 1.000 sur le train, et 0,85 en test. Un écart de 15 points. Ce modèle a donc appris le btuit du train par coeur, et non pas la structure générale du phénomène malgré le scale_pos_weight qui n'a pas pu le contenir. 






# Etape 4 Amélioration de l'approche de classification



Pour la cible (a quitte l'entreprise), il y a 4 possibilités pour un salarié

| | Le modèle prédit "reste" (0) | Le modèle prédit "part" (1) |
|---|---|---|
| **En réalité reste (0)** |  Vrai Négatif (VN) |  Faux Positif (FP) |
| **En réalité part (1)** |  Faux Négatif (FN) |  Vrai Positif (VP) |


Un algoritme de classification ne renoie pas nativement une prédiction binaire, mais revoie une probabilité d'appartenance à chaque classe (la classe des probabilité étant égal à 100%). Un seuil par défaut est défini à 50% afin trancher sur l'appartenance d'une observation à la classe 1 ou la classe 0, ce qui permet de calculé une infinité de mesures. Optiliser la rappel voudrait dire créer un algoritme avec le moins de faux négatifs possibles.

Notre contexte métier est celui d'une direction RH d'une entreprise cherchant à valoriser la situation de travail des salariés tout en limitant au maximum les dépenses.

**Qu'est-ce qui coûte le plus cher à TechNova : un Faux Négatif ou un Faux Positif ?**


Un bon modèle est un modèle qui ne commet pas d'erreurs démesurées dans ses prédictions concernant les observations les plus importantes d'un point du vue métier. Il faut se demander :
- Quel est LE périmètre de données le plus critique où le modèle doit avoir des performances optimales ?
- qu'est ce qu'on peut considérer comme une erreur acceptable d'un point de vu du client ou du décideur?
- Qu’est-ce qu’un mauvais modèle de classification dans notre contexte ?

Deux situation litigieuses :

1. Faux Négatif (FN) = le modèle dit "cette personne va rester", mais en réalité elle va démissionner. Consequence : pas d'action RH, aucune action de rétention (augmentation, entretien, mobilité interne...), et la personne part.

Coût réel : perte de compétence, coût de recrutement, perte de savoir-faire client, désorganisation des missions. C'est cher.


2. Faux Positif (FP) = le modèle dit "cette personne risque de partir", mais en réalité elle serait restée. Conséquence : les RH font un entretien de prévention, peut-être une proposition d'évolution ou une discussion RH. Coût réel : un peu de temps RH, un entretien "pour rien". C'est peu cher. pas forcément négatif.

Dans ce contexte : coût d'un faux négatif > coût d'un faux positif. Mieux vaut un modèle qui alerte trop qu'une modèle qui rate les départs. 

Métriques :

- Recall (rappel) de la classe 1 : mesure "est ce que je rate des départs réels" ? = VP / (VP + FN). Un recall élevé = peu de FN = peu de départs manqués.
- Precision de la classe 1 : mesure "quand j'alerte, ai-je raison ?" = VP / (VP + FP)
- L'Accuracy globale ne dit presque rien ici car les classes sont déséquilibrées (84% restent, 16% partent) 
- F1 = Precision * Rappel / Precision + Rappel
- ROC AUC : trace le rappel par rapport au faux positifs en fonction des seuils de classification. Le TPR est calculé comme TP / (TP + FN), c'est-à-dire la proportion d'instances positives réelles qui sont correctement identifiées par le modèle est défini par TP / (TP + FN) et le FPR est défini par FP (FP + TN). La courbe obtenue est le ROC (Receiver Operating Characteristic), elle donne une visualisation de la différence entre un bon modèle et un mauvais modèle. La métrique d'intérêt est plus précisement la surface sous la courbe ROC qui prend en compte tous les seuils de classification entre 0 et 1. Mais étant donné que le jeu de donné est déséquilibré, une courbe ROC aurait comme risque de mésestimer les faux positifs.
- Courbe PRécision (PR) qui représente sur des valeurs de présision en abscisses et le rappel en ordonné. Pour chaque seuil de possibilité, elle donne la précision et le rappel de l’algorithme en question. Cette courbe de représenter visuellement comment le modèle peut être amélioré.



Décision :

Étant donné que le coût d'un départ non anticipé (Faux Négatif) est significativement supérieur au coût d'une fausse alerte RH (Faux Positif), on privilégiera le **Recall de la classe 1** (départs) comme métrique de pilotage principale.

Sur le plan métier : une erreur de type Faux Négatif (un départ non détecté) est jugée plus coûteuse pour TechNova qu'une erreur de type Faux Positif (une fausse alerte RH suivie d'un entretien inutile).




In [16]:
sauvegarder_session(3)


Session de l'étape 3 enregistrée
Fichier : /Users/thibaultdautheville/Downloads/Projet 4/Script_intermediaire/outputs/session_etape3.pkl
Objets sauvegardés : 124
Objets ignorés : 0


PosixPath('/Users/thibaultdautheville/Downloads/Projet 4/Script_intermediaire/outputs/session_etape3.pkl')